In [1]:
import sys 
sys.path.append('C:/Users/DATA/Documents/datos/01_script/inicio/funciones')
from funciones import *
from funciones_spark import *
from variables_inicio import *
from utils_sql import *

# spark = SparkSession.builder \
#     .appName("SparkExample") \
#     .master("local[*]") \
#     .config('spark.driver.extraClassPath', 'C:/spark/jars/mssql-jdbc-13.2.1.jre11.jar') \
#     .config('spark.executor.extraClassPath', 'C:/spark/jars/mssql-jdbc-13.2.1.jre11.jar') \
#     .config('spark.executor.memory', '8g') \
#     .config('spark.driver.memory', '8g') \
#     .getOrCreate()

## actualizar retiro telef 

In [10]:
from sqlalchemy import create_engine


engine_mysql = create_engine(
    f"mysql+pymysql://{user_valentina}:{pwd_valentina}@{server_valentina}:{port_mysql}/{db_valentina}"
)

query = """
SELECT  NUMERO_DOCUMENTO,cl_telf1, cl_telf2, cl_telf3, cl_telf4, cl_telf5, cl_telf6, cl_telf7, cl_telf8, cl_telf9, cl_telf10, cl_movil, cl_celular, cl_telefono 
FROM crm_target.alfcc_clientes
WHERE cl_base = 'mayo 2026'
and cl_estado=1
"""

df_dni = pd.read_sql(query, engine_mysql)

cols_tel = [
    'cl_telf1','cl_telf2','cl_telf3','cl_telf4','cl_telf5',
    'cl_telf6','cl_telf7','cl_telf8','cl_telf9','cl_telf10',
    'cl_movil','cl_celular','cl_telefono'
]

df_long = df_dni.melt(
    id_vars='NUMERO_DOCUMENTO',
    value_vars=cols_tel,
    var_name='tipo_telf',
    value_name='CELULAR'
)
df_long['CELULAR'] = (
    df_long['CELULAR']
    .fillna(0)            
    .astype('int64')        
    .astype(str)              
)
df_long = df_long[
    (df_long['CELULAR'].notna()) &
    (df_long['CELULAR'] != '') &
    (df_long['CELULAR'].str.len() == 9) &
    (df_long['CELULAR'].str.startswith('9'))
]



In [ ]:
# UPDATE crm_target.alfcc_clientes a
# INNER JOIN crm_target.alfcc_clientes b
#     ON a.NUMERO_DOCUMENTO = b.NUMERO_DOCUMENTO
# SET 
#     a.cl_telf1 = b.cl_telf1,
#     a.cl_telf2= b.cl_telf2
# WHERE 
#     a.cl_base = 'mayo 2026'
#     AND b.cl_base <> 'abril 2026'

In [28]:
filename='20260311_BASE_CREDICASH_TARGET_TELEFONOS.csv'
filePath = os.path.join(ruta_csv, filename)

df_list = pd.read_csv(filePath,sep=';')
df_list.count()

DNI               41194
SCORE_TELEFONO    41194
CET               36861
CEL1              26912
dtype: int64

In [ ]:
df_list.rename(columns={'DNI': 'dni_cliente'}, inplace=True)


In [12]:
df_list.columns

Index(['DNI', 'SCORE_TELEFONO', 'CET', 'CEL1'], dtype='object')

In [29]:
df_list["DNI"] = (
    df_list["DNI"]
    .astype(str)
    .str.zfill(8)
)

In [22]:
df_list.head()

,DNI,SCORE_TELEFONO,CET,CEL1
0,00170492,1,979761309.0,979192639.0
1,00176167,1,986227482.0,973279919.0
2,00212754,2,984555238.0,NaN
3,44949534,3,964669369.0,999514839.0
4,03495136,2,991788185.0,NaN


In [24]:
df_list['CET'] = (
    df_list['CET']
    .astype(str)
    .str.replace(r'\.0$', '', regex=True)
    .str.strip()
)

df_list = df_list[
    (df_list['CET'].notna()) &
    (df_list['CET'] != '') &
    (df_list['CET'] != 'nan') &
    (df_list['CET'].str.len() == 9) &
    (df_list['CET'].str.startswith('9'))
]
df_list.count()

DNI               36861
SCORE_TELEFONO    36861
CET               36861
CEL1              26912
dtype: int64

In [30]:
df_list['CEL1'] = (
    df_list['CEL1']
    .astype(str)
    .str.replace(r'\.0$', '', regex=True)
    .str.strip()
)

df_list = df_list[
    (df_list['CEL1'].notna()) &
    (df_list['CEL1'] != '') &
    (df_list['CEL1'] != 'nan') &
    (df_list['CEL1'].str.len() == 9) &
    (df_list['CEL1'].str.startswith('9'))
]

In [31]:
df_list.count()

DNI               26912
SCORE_TELEFONO    26912
CET               26912
CEL1              26912
dtype: int64

In [ ]:

print(f"df_dni filas: {df_long.shape[0]}")
print(f"df_list filas: {df_list.shape[0]}")
df_list['CELULAR'] = (
    df_list['CELULAR']
    .fillna(0)            
    .astype('int64')        
    .astype(str)              
)

df_dni filas: 205154
df_list filas: 1267


In [ ]:

# telefonos = ["987863718", "991187143", "966700993", "997167060"]

# df_manual = pd.DataFrame(telefonos, columns=["TELEFONO"])


In [32]:
df_list.head()

,DNI,SCORE_TELEFONO,CET,CEL1
0,00170492,1,979761309.0,979192639
1,00176167,1,986227482.0,973279919
3,44949534,3,964669369.0,999514839
5,03496243,3,931351802.0,938240929
6,00401442,3,990771418.0,968276782


In [16]:
df_list['CELULAR'] = df_list['CELULAR'].astype(str)

df_list = df_list.merge(
    df_long,
    on="CELULAR",
    how="inner"
)

print(f"df_list filas: {df_list.shape[0]}")


df_list filas: 1697


In [18]:
df_list['retiro']='3'
df_list=df_list[['NUMERO_DOCUMENTO','retiro']]
df_list.count()

NUMERO_DOCUMENTO    1697
retiro              1697
dtype: int64

In [33]:
update_mysql_en_bloques(
    df=df_list,
    tabla="alfcc_clientes",
    periodo="mayo 2026",
    col_llave_mysql="NUMERO_DOCUMENTO",
    col_valor_mysql="cl_telf2",
    col_llave_df="DNI",
    col_valor_df="CEL1",
    host=server_valentina,
    user=user_valentina,
    password=pwd_valentina,
    database=db_valentina,
    port=port_mysql,
    batch_size=2000,
    validar_sin_grabar=False
)


Total registros a procesar: 26912
Lote 0 - 2000 actualizado | filas afectadas: 229
Lote 2000 - 4000 actualizado | filas afectadas: 211
Lote 4000 - 6000 actualizado | filas afectadas: 232
Lote 6000 - 8000 actualizado | filas afectadas: 243
Lote 8000 - 10000 actualizado | filas afectadas: 221
Lote 10000 - 12000 actualizado | filas afectadas: 232
Lote 12000 - 14000 actualizado | filas afectadas: 237
Lote 14000 - 16000 actualizado | filas afectadas: 222
Lote 16000 - 18000 actualizado | filas afectadas: 222
Lote 18000 - 20000 actualizado | filas afectadas: 222
Lote 20000 - 22000 actualizado | filas afectadas: 256
Lote 22000 - 24000 actualizado | filas afectadas: 273
Lote 24000 - 26000 actualizado | filas afectadas: 297
Lote 26000 - 26912 actualizado | filas afectadas: 127
Proceso terminado. Total filas afectadas: 3224


## actualizar retiro dni

In [20]:
from sqlalchemy import create_engine


engine_mysql = create_engine(
    f"mysql+pymysql://{user_valentina}:{pwd_valentina}@{server_valentina}:{port_mysql}/{db_valentina}"
)

query = """
SELECT 
DISTINCT 
NUMERO_DOCUMENTO as dni_cliente
FROM alfcc_clientes
WHERE cl_base = 'mayo 2026'
and cl_estado=1
"""

df_dni = pd.read_sql(query, engine_mysql)

df_dni["dni_cliente"] = (
    df_dni["dni_cliente"]
    .astype(str)
    .str.zfill(8)
)


In [23]:
filename='blacklist_dni.txt'

filePath = os.path.join(ruta_csv, filename)
df_list = pd.read_csv(filePath)
df_list.head(2)

,DNI
0,0
1,14


In [24]:

df_list.rename(columns={'DNI': 'dni_cliente'}, inplace=True)
df_list["dni_cliente"] = (
    df_list["dni_cliente"]
    .astype(str)
    .str.zfill(8)
)
print(df_list.columns)
print(df_dni.columns)


Index(['dni_cliente'], dtype='object')
Index(['dni_cliente'], dtype='object')


In [26]:
df_list = df_list.merge(
    df_dni,
    on="dni_cliente",
    how="inner"
)

print(f"df_list filas: {df_list.shape[0]}")


df_list filas: 0


In [19]:
df_list['retiro']='Retirar BlackList'
df_list=df_list[['dni_cliente','retiro']]
df_list.count()

dni_cliente    1648
retiro         1648
dtype: int64

In [ ]:
update_mysql_en_bloques(
    df=df_list,
    tabla="alfin_clientes",
    periodo="mayo 2026",
    col_llave_mysql="NUMERO_DOCUMENTO",
    col_valor_mysql="estado",
    col_llave_df="dni_cliente",
    col_valor_df="retiro",
    host=server_valentina,
    user=user_valentina,
    password=pwd_valentina,
    database=db_valentina,
    port=port_mysql,
    batch_size=2000,
    validar_sin_grabar=False
)


Total registros a procesar: 1648
Lote 0 - 1648 actualizado | filas afectadas: 0
Proceso terminado. Total filas afectadas: 0


## ejecutar query

In [ ]:
from sqlalchemy import create_engine


engine = create_engine(
    f"mysql+pymysql://{user_valentina}:{pwd_valentina}@{server_valentina}:{port_mysql}/{db_valentina}"
)
with engine.begin() as conn:
    result = conn.execute(text("""
        UPDATE crm_target.alfcc_clientes
        SET cl_estado = 3
        WHERE estado <> 'ACTIVO'
        AND cl_base = 'mayo 2026'
    """))
    
    print("Filas afectadas:", result.rowcount)

## actualizar dni a otro lote

In [3]:
filename='dni_repetidos_alfin.csv'

filePath = os.path.join(ruta_csv, filename)
df_list = pd.read_csv(filePath)

In [5]:
df_list["NUMERO_DOCUMENTO"] = (
    df_list["NUMERO_DOCUMENTO"]
    .astype(str)
    .str.zfill(8)
)
df_list.head()

,NUMERO_DOCUMENTO
0,46507674
1,40963267
2,41782588
3,40658062
4,09457398


In [6]:
df_list['lote_ref']='BD-Target -ASM'

In [7]:
update_mysql_en_bloques(
    df=df_list,
    tabla="alfin_clientes",
    periodo="mayo 2026",
    col_llave_mysql="NUMERO_DOCUMENTO",
    col_valor_mysql="lote",
    col_llave_df="NUMERO_DOCUMENTO",
    col_valor_df="lote_ref",
    host=server_valentina,
    user=user_valentina,
    password=pwd_valentina,
    database=db_valentina,
    port=port_mysql,
    batch_size=2000,
    validar_sin_grabar=False
)


Total registros a procesar: 1066
Lote 0 - 1066 actualizado | filas afectadas: 1066
Proceso terminado. Total filas afectadas: 1066


In [ ]:
BD-Target -ASM

In [5]:
exec_query_sql(server_zeus, "MAEBA", user_zeus, pwd_zeus, "ADM_OBJ_TG.spFunnelDinersTc", "SP funnel diners_tc Zeus")

SP funnel diners_tc Zeus | realizado | duración: 19.96 seg


In [ ]:
overwrite_table_SQL(spark,df_prueba_1,f'borrar_TARGET_202604_01',server_kishin,user_kishin,pwd_kishin,'DANTALION')
overwrite_table_SQL(spark,df_prueba_2,f'borrar_TARGET_202604_02',server_kishin,user_kishin,pwd_kishin,'DANTALION')
overwrite_table_SQL(spark,df_prueba_ch,f'borrar_TARGET_202604_ch',server_kishin,user_kishin,pwd_kishin,'DANTALION')


df_list filas: 1066


In [ ]:
df_dni filas: 128183
df_list filas: 12546

In [ ]:
ssss

In [7]:
print(df_list.columns)
print(df_long.columns)

Index(['TELEFONO'], dtype='object')
Index(['NUMERO_DOCUMENTO', 'tipo_telf', 'TELEFONO'], dtype='object')


df_list filas: 1


In [ ]:
df_list filas: 2036

NUMERO_DOCUMENTO    1
retiro              1
dtype: int64

In [10]:
df_list.head()

,NUMERO_DOCUMENTO,retiro
0,40503275,Retirar Telef


Total registros a procesar: 1
Lote 0 - 1 actualizado | filas afectadas: 1
Proceso terminado. Total filas afectadas: 1
